In [24]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from sklearn.preprocessing import MinMaxScaler

In [25]:
df = pd.read_csv('../../data/airpassengers/AirPassengers.csv')
data = df['#Passengers'].values.astype(float)

In [26]:
print(data[:5])
print(data.shape)

[112. 118. 132. 129. 121.]
(144,)


In [27]:
# LSTM requires proper scaling
# reshape changes from 1 row of data, to 1 column of data, (-1,1) means automatic no. of rows and 1 column
# x_scaled = (x - min) / (max - min) is the method that fit_transform uses to scale data
# flatten squashes it back to 1 row

scaler = MinMaxScaler()
data = scaler.fit_transform(data.reshape(-1, 1)).flatten()

print(data[:5])

[0.01544402 0.02702703 0.05405405 0.04826255 0.03281853]


In [28]:
def make_sequences(data, seq_len=12):   # seq_len means how long is the sequence of data before my next prediction
    X, y = [], []   # X will handle input_window, y will handle to targets
    for i in range(len(data) - seq_len):    # we are sliding across by 1, so [1,13], [2,14], ... instead of [1,13], [14, 27], ...
        X.append(data[i : i+seq_len])       # 12 months in
        y.append(data[i + seq_len])         # predict month 13
    return (torch.tensor(X, dtype=torch.float32),
            torch.tensor(y, dtype=torch.float32))

X, y = make_sequences(data, seq_len=12)
X = X.unsqueeze(-1)   # (132, 12, 1)  ← LSTM needs (batch, time, features)

print(X.shape)
print(y.shape)

torch.Size([132, 12, 1])
torch.Size([132])


In [29]:
split = int(len(X) * 0.8)

X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(X_train.shape)  # → (105, 12, 1)
print(X_test.shape)   # → (27, 12, 1)

torch.Size([105, 12, 1])
torch.Size([27, 12, 1])


In [30]:
class LSTMForecaster(nn.Module):
    def __init__(self, input_size=1, hidden_size=64, num_layers=2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size,         # input_size is how many features per timestep
            hidden_size,        # hidden_size is how many memory units the LSTM has, bigger == more ability to learn patterns (working memory width), normally hidden_size is around input_size
            num_layers,         # num_layers is how many times to repeat the LSTM
            batch_first=True,   # just puts batch first, x shape = (batch, time, features) = (32, 12, 1), also batch here is the batch of data fed from DataLoader
            dropout=0.2         # at the end of LSTM
        )
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        out, _ = self.lstm(x)       # out: (B, T, hidden_size)
        last = out[:, -1, :]        # take last timestep only: (B, hidden_size)
        return self.fc(last).squeeze()

model = LSTMForecaster()

In [31]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.MSELoss()    # Mean square error used for Regression task (how far value is off)

for epoch in range(200):
    model.train()
    optimizer.zero_grad()

    pred = model(X_train)
    loss = criterion(pred, y_train)

    loss.backward()
    optimizer.step()

    if (epoch + 1) % 50 == 0:
        model.eval()
        with torch.no_grad():
            test_loss = criterion(model(X_test), y_test)
        print(f"Epoch {epoch+1}  train={loss:.4f}  test={test_loss:.4f}")

Epoch 50  train=0.0229  test=0.0954
Epoch 100  train=0.0071  test=0.0213
Epoch 150  train=0.0055  test=0.0199
Epoch 200  train=0.0054  test=0.0182


In [32]:
model.eval()
with torch.no_grad():
    preds = model(X_test).numpy().reshape(-1, 1)
    preds = scaler.inverse_transform(preds)        # ← unnormalise predictions
    actuals = scaler.inverse_transform(
        y_test.numpy().reshape(-1, 1)              # ← unnormalise actuals too
    )

for p, a in zip(preds[:5], actuals[:5]):
    print(f"predicted: {p[0]:.0f}   actual: {a[0]:.0f}")

predicted: 391   actual: 359
predicted: 393   actual: 310
predicted: 399   actual: 337
predicted: 402   actual: 360
predicted: 405   actual: 342
